# Data Preprocess 

This script concatenates most recent passive data with backup data and performs basic preprocessing on passive, EMA and Monitoring data.

In [ ]:
from pyprojroot import here # define relative paths to the project root (working directory)
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc  
import os
import glob
import numpy as np
import pickle
import plotly.express as px

# --- Paths / imports -------------------------------------------------

# relative project root
PROJECT_ROOT = here() # '.here' is located as invisible file in the project root working directory
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path

from functions.preprocessing.infer_timeoffset import (
    create_utcday_tzoffset_df,
    merge_fill_tz,
)
# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")        
today_day = pd.Timestamp.today().normalize()       

# --- Path -------------------------------------------------------------
today_str = "04052026"
datapath = Path(raw_path) / f"export_tiki_{today_str}"  

## 1. Passive Data

### 1.1 Load most recent passive data

In [ ]:
# actual passive + ema_data

# define the pattern for passive data files
file_pattern = os.path.join(datapath, "epoch_part*.csv")

# use glob to find all matching files
file_list = glob.glob(file_pattern)

# sort the file list for consistent ordering
file_list.sort()

# concatenate all CSV files into a single DataFrame
df_complete = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)


In [ ]:
# Extract customer identifier and reduce to first 4 characters
df_complete["customer"] = df_complete.customer.str.split("@").str.get(0)
df_complete["customer"] = df_complete["customer"].str[:4]

for col in ["startTimestamp", "endTimestamp"]:
    df_complete[col] = (
        pd.to_datetime(df_complete[col], utc=True, errors="coerce", unit="ms")
    )

#df_complete

In [ ]:
df_complete = df_complete[['customer', 'startTimestamp', 'endTimestamp', 'timezoneOffset', 'type',
       'stringValue', 'booleanValue', 'doubleValue', 'longValue']]

### 1.2 Load big backup dataset

In [ ]:
# merge with backup data
backup_path = preprocessed_path + "/backup_passive_05052025.parquet"

# backup_path = preprocessed_path + "/backup_passive_last.feather"
df_backup = pd.read_parquet(backup_path)

In [ ]:
# make independent copies of both DataFrames to avoid SettingWithCopyWarning (future modifications do not affect any original DataFrame)
df_backup = df_backup.copy()
df_complete = df_complete.copy()

# convert booleanValue to boolean[pyarrow] dtype before concatenation so that it can be saved to .feather later
# alternative to "boolean[pyarrow]" is "boolean", but it is experimental and may change in future pandas versions
df_backup['booleanValue'] = df_backup['booleanValue'].astype('boolean[pyarrow]')
df_complete['booleanValue'] = df_complete['booleanValue'].astype('boolean[pyarrow]')

In [ ]:
# latest timestamp from the backup dataset
latest_timestamp = df_backup['startTimestamp'].max()

# filter from df_complete only those entries that are newer than what’s already in the backup
df_complete_filtered = df_complete[df_complete['startTimestamp'] > latest_timestamp]

### 1.3 Concat Backup and most recent data

In [ ]:
# update the backup by concatenating only the newly filtered rows from df_complete, creating an up-to-date backup
df_backup_recent = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

### 1.4 Rename variable names and create additional columns 

In [ ]:
# define a clear mapping for rename columns
rename_columns = {"customer": "id",
                  "type": "modality",
                  "startTimestamp": "timestamp_start",
                  "endTimestamp": "timestamp_end",
                  "booleanValue": "boolean_value",
                  "doubleValue": "double_value",
                  "longValue": "long_value",
                  "timezoneOffset": "timezone_offset"}

# apply renaming
df_backup_recent = df_backup_recent.rename(columns=rename_columns)

In [ ]:
# create a unified float_value column:
# use 'doubleValue' where available (more precise), otherwise use 'longValue'
df_backup_recent['float_value'] = df_backup_recent['double_value'].fillna(df_backup_recent['long_value'])


In [ ]:
# drop original value columns that have been unified into 'float_value' + 'stringValue' (because only ECG data are stored as string for period March - November 2023) + 'createdtAt'
df_backup_recent = df_backup_recent.drop(columns=['double_value', 'long_value', 'stringValue'])


In [ ]:
# create a time_interval (duration in seconds) column
df_backup_recent['time_interval'] = (
    df_backup_recent['timestamp_end'] - df_backup_recent['timestamp_start']
).dt.total_seconds()

# create a start_date and start_hour column
df_backup_recent['start_date']  = df_backup_recent['timestamp_start'].dt.normalize()
df_backup_recent['start_hour'] = df_backup_recent['timestamp_start'].dt.hour

### 1.5 Infer Timezone Offset

In [ ]:
df_tz = create_utcday_tzoffset_df(df_backup_recent)


In [ ]:
df_tz["inferred_tzoffset_timedelta"] = pd.to_timedelta(
    df_tz["inferred_tzoffset"], unit="min"
)

In [ ]:
df_tz.columns

In [ ]:
df_backup_recent = df_backup_recent.merge(
    df_tz,
    left_on=["id", "start_date"],
    right_on=["id", "day"],
    how="left",
)
#df_backup_recent.drop(columns=["day"], inplace=True)  # remove day from df_tz

In [ ]:
df_backup_recent["local_timestamp_start"] = (
    df_backup_recent["timestamp_start"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent["local_timestamp_end"] = (
    df_backup_recent["timestamp_end"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent.head()

In [ ]:
assert df_backup_recent.inferred_tzoffset.isna().sum() == 0, (
    "There are missing inferred timezone offsets!"
)

In [ ]:
# just to make sure that we don't use them anymore later
del df_complete_filtered
del df_complete

Next:

1. Since we want to include 'for_id' and 'study_version' in our `passive_data` data frame, we need to extract these data from the monitoring sheet. This is done in section 2

2. Additionally the `monitoring_data` data frame is set up in section 2


## 2. Monitoring data

In [ ]:
# import data
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [ ]:
# get an overview of the monitoring data
#df_monitoring.head()

In [ ]:
# show columns of monitoring data
print(df_monitoring.columns.tolist())

In [ ]:
df_monitoring = df_monitoring.copy()

df_monitoring.rename(columns = {"Pseudonym": "id",
                                "FOR_ID": "for_id",
                                "EMA_ID": "ema_id", 
                                "Status": "study_status",
                                "Studienversion":"study_version", 
                                "Start EMA Baseline": "ema_base_start", 
                                "Ende EMA Baseline": "ema_base_end", 
                                "Freischaltung/ Start EMA T20": "ema_t20_start",
                                "Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", 
                                "T20=Post":"t20_post" }, 
                     inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'id', 'study_version', 'study_status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end']]

df_monitoring["id"] = df_monitoring["id"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)


### 2.1 Merge relevant columns with passive data

In [ ]:
df_monitoring_short = df_monitoring[["id", "for_id","study_version"]]

#### 2.2 Final `passive_data` data frame

In [ ]:
df_backup_recent = df_backup_recent.merge(df_monitoring_short, on="id", how="right")

In [ ]:

manual_for_map = {
    "asYVE": "FOR11011",
    "DWmH": "FOR11015",
    "dT0F": "FOR13008",
    "yv0Q": "FOR13012",
    "w0Ep": "FOR14051",
    "P0Tz": "FOR11156",
}

df_backup_recent["for_id"] = df_backup_recent["for_id"].fillna(df_backup_recent["id"].map(manual_for_map))

In [ ]:
df_backup_recent.head()

In [ ]:
# ensure data types are coded correctly
df_backup_recent['boolean_value'] = df_backup_recent['boolean_value'].astype('boolean[pyarrow]')
df_backup_recent['study_version'] = df_backup_recent['study_version'].astype('string')
df_backup_recent['modality'] = df_backup_recent['modality'].astype('string')
df_backup_recent['id'] = df_backup_recent['id'].astype('category')

In [ ]:
# Get a list of columns to drop (all columns not in keep_cols)
keep_cols_passive = ['id','for_id', 'modality', 'timestamp_start','timestamp_end',
    'local_timestamp_start', 'local_timestamp_end','time_interval', 'float_value', 'boolean_value','start_date', 
    'start_hour', "timezone_offset", 'study_version']

# final passive data frame
df_passive_final = df_backup_recent[keep_cols_passive]

#### 2.3 Final `monitoring_data` data frame

In [ ]:
#??

## 3. EMA data

#### 3.1 Load, match and rename relevant data from separate .csv files

In [ ]:

# Beispiel: datapath = Path("/pfad/zum/verzeichnis")
session        = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers        = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice         = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions      = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire  = pd.read_csv(datapath / "questionnaires.csv", low_memory=False)  # ohne Komma!


In [ ]:
# session data
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 
session["user"] = session["user"].str[:4]
session.rename(columns = {"id":"session_id",
                          "user":"id",
                          "completedAt": "timestamp_beep_completion", 
                          "createdAt": "timestamp_item_completion", 
                          "expirationTimestamp": "timestamp_beep_expiration",
                          "sessionRun":"beep_num_run",
                          "study":"schedule_chronotype"}, inplace=True)
session['measurement_burst'] = session['schedule_chronotype'].map(study_mapping)
session['schedule_chronotype'] = session['schedule_chronotype'].map(chronotype_mapping)
# Parse epoch‑ms columns as UTC and drop tz info -> naive UTC
for col in ["timestamp_item_completion", "timestamp_beep_completion", "timestamp_beep_expiration"]:
    session[col] = (
        pd.to_datetime(session[col], unit="ms", utc=True, errors="coerce")
    )

df_sess = session[["id","session_id", "measurement_burst", "beep_num_run", "timestamp_item_completion", "timestamp_beep_completion", "schedule_chronotype", "timestamp_beep_expiration"]]

In [ ]:
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 

answers["user"] = answers["user"].str[:4]
answers = answers[["user", "questionnaire", "study", "question", "element",
                   "createdAt", "order", "questionnaireSession"]]

answers["createdAt"] = (
    pd.to_datetime(answers["createdAt"], unit="ms", utc=True, errors="coerce")
)
answers['measurement_burst'] = answers['study'].map(study_mapping)
answers['schedule_chronotype'] = answers['study'].map(chronotype_mapping)

answers.rename(columns={"user": "id", 
                        "createdAt": "timestamp_item_completion",
                        "questionnaire": "beep_type",
                        "question":"item_code_map",
                        "order":"item_order",
                        "questionnaireSession":"session_id",
                        }, inplace=True)
answers.drop(columns=["study"], inplace=True)

In [ ]:
# item description data
choice = choice[["element", "choice_id", "text", "question"]]
choice.rename(columns={"text":"response_text",
                       "choice_id":"response", 
                       "question":"item_code_map"}, inplace=True)

In [ ]:
# question description data
questions = questions[["id", "title"]]
questions.rename(columns={"id":"item_code_map",
                          "title":"item"}, inplace=True)

In [ ]:
questionnaire = questionnaire[["id", "name"]]
questionnaire.rename(columns={"id":"beep_type",
                              "name":"beep_type_name"}, inplace=True)

In [ ]:
answer_merged = pd.merge(answers, choice, on= ["item_code_map","element"])
answer_merged = pd.merge(answer_merged, questions, on= "item_code_map")
answer_merged = pd.merge(answer_merged, questionnaire, on= "beep_type")
answer_merged["date"] = answer_merged["timestamp_item_completion"].dt.normalize()

#### 3.2 Calculate auxiliary variables

In [ ]:
df_monitoring_ema = df_monitoring[["id", "for_id","study_version", 'ema_base_start', 'ema_base_end',
       'ema_t20_start', 'ema_t20_end', 'ema_post_start', 'ema_post_end', 't20_post']]

In [ ]:
bursts = [("base", 0), ("t20", 1), ("post", 2)]

out = []
for burst_name, burst_code in bursts:
    tmp = df_monitoring_ema[[
        "id", "for_id", "study_version", "t20_post",
        f"ema_{burst_name}_start", f"ema_{burst_name}_end"
    ]].copy()

    tmp = tmp.rename(columns={
        f"ema_{burst_name}_start": "ema_burst_start",
        f"ema_{burst_name}_end":   "ema_burst_end",
    })
    tmp["measurement_burst"] = burst_code
    out.append(tmp)

df_monitoring_ema_long = (
    pd.concat(out, ignore_index=True)
      # optional: drop rows where the burst is entirely missing
      .dropna(subset=["ema_burst_start", "ema_burst_end"], how="all")
      .sort_values(["id", "measurement_burst"])
      .reset_index(drop=True)
)


In [ ]:
answer_merged = pd.merge(answer_merged, df_monitoring_ema_long, on = ["id", "measurement_burst"])

In [ ]:
df_ema_content = answer_merged.copy()

In [ ]:
meta_cols = ['id','for_id','date','response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order', 'session_id', 'measurement_burst']
df_ema_meta = df_ema_content[meta_cols].copy()

In [ ]:
df_sess_short = df_sess[["id", "session_id",  "beep_num_run",'timestamp_beep_completion']].copy()

In [ ]:
df_ema_meta = df_ema_meta.merge(df_sess_short, on=["id", "session_id"], how="left")

#### 3.3 Calculate EMA coverage

In [ ]:
df_sess_short_compliance = df_sess[["id", "session_id", 'timestamp_beep_completion']].copy()

In [ ]:
df_ema_content = df_ema_content.merge(df_sess_short_compliance, on=["id", "session_id"], how="left")

In [ ]:
df_ema_content["n_beeps_beeps_completed_per_burst"] = (
    df_ema_content
    .groupby(["measurement_burst", "id"])["timestamp_beep_completion"]
    .transform("nunique"))

#### 3.3 Calculate auxiliary variables

In [ ]:
#df_ema_content = answer_merged.copy()

In [ ]:
# 1. Date and Time Manipulations
df_ema_content['weekday'] = df_ema_content['timestamp_item_completion'].dt.day_name()


# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    elif month in [9, 10, 11]:
        return 3
    else:
        return 4

df_ema_content['season'] = df_ema_content['timestamp_item_completion'].dt.month.apply(get_season)

# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return 1
    elif 8 <= hour < 12:
        return 2
    elif 12 <= hour < 17:
        return 3
    elif 17 <= hour < 21:
        return 4
    else:
        return 5

df_ema_content['time_of_day'] = df_ema_content['timestamp_item_completion'].dt.hour.apply(get_time_of_day)
df_ema_content['item'] = df_ema_content['item'].str.replace('_morning', '', regex=False)

# 3. Weekend Indicator
df_ema_content['weekend'] = df_ema_content['weekday'].isin(['Saturday', 'Sunday']).astype(int)

# 4. Questionnaire Number
df_ema_content['nr_beep_daily'] = df_ema_content['beep_type_name'].str.extract(r'(\d+)').astype(float)

# 5. Count unique questionnaires per day
df_ema_content['n_beeps_completed_per_day'] = (
    df_ema_content.groupby(['measurement_burst', 'id', 'date'])['beep_type_name']
                  .transform('nunique')
)

# 6. Unique Day Identifier
df_ema_content['quest_nr_str'] = df_ema_content['nr_beep_daily'].fillna('unknown').astype(str)
df_ema_content['beep_per_person_id'] = (
    df_ema_content['date'].dt.strftime('%Y%m%d') + '_' + df_ema_content['quest_nr_str']
)

In [ ]:
# --- manual exception: ID mASG ---
# reassign misclassified burst 2 data to burst 1 

start_fix = pd.Timestamp("2025-08-21", tz="UTC")
end_fix   = pd.Timestamp("2025-09-05", tz="UTC")

mask_mASG_fix = (
    (df_ema_content["id"] == "mASG") &
    (df_ema_content["measurement_burst"] == 2) &
    (df_ema_content["date"].between(start_fix, end_fix))
)

df_ema_content.loc[mask_mASG_fix, "measurement_burst"] = 1


In [ ]:
# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content['ema_relative_start'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('min')
)
df_ema_content['ema_relative_end'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('max')
)

# 8. Absolute & Relative Day Index
df_ema_content['absolute_day_index'] = (
    df_ema_content['date'] - df_ema_content['ema_relative_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date']
                  .rank(method='dense').astype(int)
)


In [ ]:
# 9. Filter absolute_day_index > 16
max_allowed_days = 16
#df_ema_content = df_ema_content[df_ema_content['absolute_day_index'] <= max_allowed_days].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
high_indices_id = high_indices.id.unique().tolist()
if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['id'].unique())
else:
    print("All entries have absolute_day_index <= 16.")



In [ ]:
df_max_day = (
    df_ema_content.loc[
        df_ema_content["id"].isin(high_indices_id) & (df_ema_content["absolute_day_index"] > 16),
        ["id", "measurement_burst", "ema_relative_start", "ema_relative_end", "absolute_day_index", "beep_per_person_id"]
    ]
    .groupby(["id", "measurement_burst", "ema_relative_start", "ema_relative_end"], as_index=False)
    .agg(
        max_absolute_day_index=("absolute_day_index", "max"),
        n_unique_beeps_after_day16=("beep_per_person_id", "nunique"),
    )
    .sort_values(["id", "measurement_burst", "max_absolute_day_index"])
)

df_max_day_burst0 = df_max_day.loc[df_max_day["measurement_burst"] == 0]
df_max_day_burst1 = df_max_day.loc[df_max_day["measurement_burst"] == 1]
df_max_day_burst2 = df_max_day.loc[df_max_day["measurement_burst"] == 2]

df_max_day_burst1

CLUSTER PLOTTING TO DECIDE HOW TO PROCEED WITH MAX DAY INDEX > 16
* spot days before/after the intended 16 day window
* gaps or irregular participation
* whether 'extra' days are meaningful or just noise

In [ ]:
def plot_ema_clusters(
    df_ema_content,
    df_monitoring,
    df_ids_to_keep,
    burst_number,
    start_col,
    monitoring_id_col="id",
    ema_id_col="id",
    burst_col="measurement_burst",
    date_col="date"
):
    
    """
    Plot daily EMA beep counts per participant for a given measurement burst,
    with dashed lines for:
    - day 16 after first observed activation
    - scheduled EMA start date from df_monitoring
    Parameters
    ----------
    df_ema_content : pd.DataFrame
        EMA content dataframe.
    df_monitoring : pd.DataFrame
        Monitoring dataframe containing scheduled start dates.
    df_ids_to_keep : pd.DataFrame
        Dataframe containing IDs to retain (e.g. df_max_day_burst1).
    burst_number : int
        Measurement burst to plot (e.g. 1 or 2).
    start_col : str
        Column in df_monitoring with the scheduled EMA start date.
    monitoring_id_col : str
        ID column in df_monitoring before shortening.
    ema_id_col : str
        ID column in df_ema_content and df_ids_to_keep.
    burst_col : str
        Burst column in df_ema_content.
    date_col : str
        Date column in df_ema_content.
    """

    # --- Step 1: filter ema data ---
    df_ema = df_ema_content.copy()
    df_ema[date_col] = pd.to_datetime(df_ema[date_col], errors="coerce")
    df_ema[ema_id_col] = df_ema[ema_id_col].astype(str).str.strip()

    ids_to_keep = (
        df_ids_to_keep[ema_id_col]
        .astype(str)
        .str.strip()
        .drop_duplicates()
    )

    filtered = df_ema[
        (df_ema[burst_col] == burst_number) &
        (df_ema[ema_id_col].isin(ids_to_keep))
    ].copy()

    filtered = filtered.sort_values([ema_id_col, date_col])

    filtered["day_in_burst"] = (
        filtered.groupby(ema_id_col)[date_col]
        .transform(lambda x: (x - x.min()).dt.days + 1)
    )

   # --- Step 2: prepare scheduled starts --- 
    monitoring = df_monitoring.copy()
    monitoring[monitoring_id_col] = (
        monitoring[monitoring_id_col].astype(str).str.split("@").str.get(0).str[:4].str.strip()
    )
    monitoring[start_col] = pd.to_datetime(monitoring[start_col], errors="coerce")

    start_map = (
        monitoring[[monitoring_id_col, start_col]]
        .dropna(subset=[monitoring_id_col])
        .drop_duplicates(subset=[monitoring_id_col])
        .rename(columns={monitoring_id_col: ema_id_col})
    )

    # --- Step 3: merge onto filtered burst and id ---
    filtered = filtered.merge(start_map,on=ema_id_col,how="left")

    # --- Step 4: count beeps per day per id ---
    daily_counts = (
        filtered
        .groupby([ema_id_col, date_col, "day_in_burst", start_col], dropna=False)
        .size()
        .reset_index(name="n_beeps")
    )

    # --- Step 5: plot one figure per id ---
    for pid, sub in daily_counts.groupby(ema_id_col):
        fig = px.scatter(
            sub,
            x=date_col,
            y="n_beeps",
            title=f"id {pid} | burst {burst_number}",
            labels={
                date_col: "Date",
                "n_beeps": "Number of beeps"
            }
        )

        # black dashed line: 16th day after first observed activation
        cutoff_date = sub[date_col].min() + pd.Timedelta(days=15)
        fig.add_vline(
            x=cutoff_date,
            line_dash="dash"
        )

        # orange dashed line: scheduled EMA start
        ema_start = sub[start_col].iloc[0]
        if pd.notna(ema_start):
            fig.add_vline(
                x=ema_start,
                line_dash="dash",
                line_color="orange"
            )

        fig.show()

    return daily_counts, filtered

In [ ]:
# OUT OF PHASE ACTIVATIONS T20 (IDs)
df_max_day_burst1 

In [ ]:
# CLUSTER PLOTTING T20
daily_counts_b1, filtered_b1 = plot_ema_clusters(
    df_ema_content=df_ema_content,
    df_monitoring=df_monitoring,
    df_ids_to_keep=df_max_day_burst1,
    burst_number=1,
    start_col="ema_t20_start"
)

In [ ]:
# OUT OF PHASE ACTIVATIONS TPost (IDs)
df_max_day_burst2

In [ ]:
# CLUSTER PLOTTING TPost
daily_counts_b2, filtered_b2 = plot_ema_clusters(
    df_ema_content=df_ema_content,
    df_monitoring=df_monitoring,
    df_ids_to_keep=df_max_day_burst2,
    burst_number=2,
    start_col="ema_post_start"
)

In [ ]:
# remove beeps after day 16 for the measurement burst 0
df_ema_content = df_ema_content.loc[
    ~(
        (df_ema_content["measurement_burst"] == 0) &
        (df_ema_content["absolute_day_index"] > 16)
    )
].copy()

In [ ]:
# remove out of phase data for measurement burst 1 and 2 (self-activation was possible, leading to false-activation of study phases)
# rule: keep all data from scheduled start (T20 and TPost) counting upwards to day 16, cut everything else
# exception 1: one case in which the last day with recorded ema beeps ends one day before scheduled start -> decided to keep (t20)
# exception 2: one case in which ema recording started a few days after offically scheduled start -> decided to keep (t20)
# exception 3: one case in which ema recording was self-activated one day before scheduled start -> decided to keep this day (tpost)

# keep raw data unchanged
df_ema_clean = df_ema_content.copy()

# flagged IDs only 
ids_burst1 = set(df_max_day_burst1["id"].astype(str).str.strip())
ids_burst2 = set(df_max_day_burst2["id"].astype(str).str.strip())

# prepare EMA data 
df_ema_clean["id"] = df_ema_clean["id"].astype(str).str.strip()
df_ema_clean["date"] = (pd.to_datetime(df_ema_clean["date"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize())

# prepare monitoring data
df_monitoring_clean = df_monitoring.copy()
df_monitoring_clean["id_short"] = (df_monitoring_clean["id"].astype(str).str.split("@").str[0].str[:4].str.strip())
df_monitoring_clean["ema_t20_start"] = (pd.to_datetime(df_monitoring_clean["ema_t20_start"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize())
df_monitoring_clean["ema_post_start"] = (pd.to_datetime(df_monitoring_clean["ema_post_start"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize())

# merge scheduled starts onto EMA data
df_ema_clean = df_ema_clean.merge(
    df_monitoring_clean[["id_short", "ema_t20_start", "ema_post_start"]]
    .drop_duplicates("id_short")
    .rename(columns={"id_short": "id"}),
    on="id",
    how="left"
)

# manual exceptions 
exception_id_burst1_before_start = "fx8B"
exception_id_burst1_keep_after_start = "p1zH"
exception_id_post = "p1zH"

# define keep rules
keep_burst1 = (
    # normal rule: T20 start until day 16
    df_ema_clean["date"].between(
        df_ema_clean["ema_t20_start"],
        df_ema_clean["ema_t20_start"] + pd.Timedelta(days=15)
    )
    |
    # exception fx8B: keep data before T20 start
    (
        (df_ema_clean["id"] == exception_id_burst1_before_start) &
        (df_ema_clean["date"] < df_ema_clean["ema_t20_start"])
    )
    |
    # exception p1zH: keep all T20/burst 1 data after scheduled T20 start
    (
        (df_ema_clean["id"] == exception_id_burst1_keep_after_start) &
        (df_ema_clean["date"] >= df_ema_clean["ema_t20_start"])
    )
)


# manual exception for burst 2 
keep_burst2 = (
    df_ema_clean["date"].between(
        df_ema_clean["ema_post_start"],
        df_ema_clean["ema_post_start"] + pd.Timedelta(days=15)
    )
    |
    # exception p1zH: keep exactly one day before scheduled TPost start
    (
        (df_ema_clean["id"] == exception_id_post) &
        (df_ema_clean["date"] == df_ema_clean["ema_post_start"] - pd.Timedelta(days=1))
    )
)


# explanation: keep the normal TPost window plus exactly one day before scheduled TPost start for p1zH

# --- apply cut only to flagged IDs ---
df_ema_clean = df_ema_clean.loc[
    ~(((df_ema_clean["measurement_burst"] == 1) & (df_ema_clean["id"].isin(ids_burst1)) & ~keep_burst1)
        |
        ((df_ema_clean["measurement_burst"] == 2) & (df_ema_clean["id"].isin(ids_burst2)) & ~keep_burst2)
    )
].copy()


# optional: drop scheduled start columns again
df_ema_clean = df_ema_clean.drop(columns=["ema_t20_start", "ema_post_start"])



In [ ]:
# sanity check: compare rows and IDs before vs after cut
def summarize_cut_t20_post(before_df, after_df, burst_col="measurement_burst", id_col="id", bursts=(1, 2)):
    before = before_df.copy()
    after = after_df.copy()

    before[id_col] = before[id_col].astype(str).str.strip()
    after[id_col] = after[id_col].astype(str).str.strip()

    before = (before[before[burst_col].isin(bursts)]
        .groupby(burst_col)
        .agg(rows_before=(id_col, "size"),
            ids_before=(id_col, "nunique")))

    after = (after[after[burst_col].isin(bursts)]
        .groupby(burst_col)
        .agg(rows_after=(id_col, "size"),
            ids_after=(id_col, "nunique")))

    summary = pd.concat([before, after], axis=1).fillna(0).astype(int)
    summary["rows_removed"] = summary["rows_before"] - summary["rows_after"]
    summary["ids_removed"] = summary["ids_before"] - summary["ids_after"]

    return summary.reset_index()

summary = summarize_cut_t20_post(df_ema_content, df_ema_clean)
print(summary)

In [ ]:
# please double check if the removal of T20 and TPost out of phase activation is running correctly (therefore run everything safely on df_ema_clean): 
# check if the table summary and clustering is consistent, if not adjust the code.

# if everything is working properly change 'df_ema_clean' to 'df_ema_content' so that the notebook continues with the cleaned df
df_ema_content = df_ema_clean.copy()

In [ ]:
# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(subset=['id', 'measurement_burst', 'beep_per_person_id']).copy()
df_unique['relative_beep_counter'] = df_unique.groupby(['id', 'measurement_burst']).cumcount() + 1
df_ema_content = df_ema_content.merge(
    df_unique[['id', 'measurement_burst', 'beep_per_person_id', 'relative_beep_counter']],
    on=['id', 'measurement_burst', 'beep_per_person_id'],
    how='left'
)

# 12. Missing Data behandeln
df_ema_content['measurement_burst'] = df_ema_content['measurement_burst'].fillna('unknown')
df_ema_content['absolute_day_index'] = df_ema_content['absolute_day_index'].where(
    df_ema_content['ema_relative_start'].notna(), np.nan
)

### 3.4 merge the inferred tz offsets

In [ ]:
# uncomment if want to run this cell multiple times
# if "infered_tzoffset" in df_sess.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_sess")
#     df_sess = df_sess.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source", "inferred_tzoffset_timedelta"])
# if "inferred_tzoffset" in df_ema_meta.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_ema_meta")
#     df_ema_meta = df_ema_meta.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source", "inferred_tzoffset_timedelta"])
# if "inferred_tzoffset" in df_ema_content.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_ema_content")
#     df_ema_content = df_ema_content.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source", "inferred_tzoffset_timedelta"])


df_ema_meta = merge_fill_tz(
    df_ema_meta, df_tz, day_col="date", customer_col="id"
)
# df_tz expects the date column to be timezone-aware (UTC), so we need to localize it before merging
df_ema_content["date"] = df_ema_content["date"].dt.tz_localize("utc")
df_ema_content = merge_fill_tz(
    df_ema_content, df_tz, day_col="date", customer_col="id"
)

In [ ]:

df_ema_content.drop(columns=['response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order', 'session_id'], inplace=True) 

### Export passive, EMA and Monitoring

In [ ]:
backup_path = raw_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(backup_path)

preprocessed_path_final = preprocessed_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(preprocessed_path_final)

#preprocessed_path_freezed_final = preprocessed_path_freezed + "/code_check" + "/backup_passive_recent.parquet"
#df_passive_final.to_parquet(preprocessed_path_freezed_final)

with open(preprocessed_path + '/ema_meta.pkl', 'wb') as file:
    pickle.dump(df_ema_meta, file)

    
with open(preprocessed_path + '/monitoring_data.pkl', 'wb') as file:
    pickle.dump(df_monitoring, file)

    
with open(preprocessed_path + '/ema_content.pkl', 'wb') as file:
    pickle.dump(df_ema_content, file)

In [ ]:

# Export ema meta as CSV
df_ema_path = preprocessed_path + '/ema_meta.csv'
df_ema_meta.to_csv(df_ema_path, index=False)

# Export df_monitoring as CSV
df_monitoring_csv_path = preprocessed_path + '/monitoring_data.csv'
df_monitoring.to_csv(df_monitoring_csv_path, index=False)

# Export df_ema_content as CSV
df_ema_content_csv_path = preprocessed_path + '/ema_content.csv'
df_ema_content.to_csv(df_ema_content_csv_path, index=False)

# Export df_ema_content as CSV to freezed for data check
#df_ema_content_csv_path = preprocessed_path_freezed +'/code_check' +'/ema_content_recent.csv'
#df_ema_content.to_csv(df_ema_content_csv_path, index=False)

